# 05 - Detección de anomalías transaccionales con Isolation Forest

## Caso fintech
Este notebook identifica transacciones anómalas sin usar labels.

## Objetivo
Detectar transacciones fuera del patrón normal de comportamiento.

## Dataset
Se genera un dataset sintético de 4,000 transacciones.

## Técnicas
- Aprendizaje no supervisado
- Isolation Forest
- Scoring de anomalías
- Visualización de outliers


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


In [ ]:
n_normal = 3900
n_anomalies = 100

normal_amount = np.random.lognormal(mean=3.3, sigma=0.7, size=n_normal)
normal_freq = np.random.poisson(4, n_normal)
normal_distance = np.random.exponential(10, n_normal)
normal_merchant_risk = np.random.beta(2, 8, n_normal)

anomaly_amount = np.random.lognormal(mean=5.2, sigma=0.9, size=n_anomalies)
anomaly_freq = np.random.poisson(14, n_anomalies)
anomaly_distance = np.random.exponential(80, n_anomalies)
anomaly_merchant_risk = np.random.beta(6, 2, n_anomalies)

df = pd.DataFrame({
    "amount": np.concatenate([normal_amount, anomaly_amount]).round(2),
    "transactions_last_24h": np.concatenate([normal_freq, anomaly_freq]),
    "distance_from_home_km": np.concatenate([normal_distance, anomaly_distance]).round(2),
    "merchant_risk_score": np.concatenate([normal_merchant_risk, anomaly_merchant_risk]).round(3),
    "synthetic_label": np.concatenate([np.zeros(n_normal), np.ones(n_anomalies)]).astype(int)
})

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()


In [ ]:
df.shape, df["synthetic_label"].value_counts()

In [ ]:
features = ["amount", "transactions_last_24h", "distance_from_home_km", "merchant_risk_score"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

model = IsolationForest(
    n_estimators=200,
    contamination=0.025,
    random_state=42
)

df["anomaly_prediction"] = model.fit_predict(X_scaled)
df["anomaly_score"] = model.decision_function(X_scaled)

# Isolation Forest devuelve -1 para anomalía y 1 para normal
df["is_anomaly"] = (df["anomaly_prediction"] == -1).astype(int)

df.head()


In [ ]:
pd.crosstab(df["synthetic_label"], df["is_anomaly"], rownames=["Real sintético"], colnames=["Predicción anomalía"])

In [ ]:
plt.figure(figsize=(8, 6))
normal = df[df["is_anomaly"] == 0]
anomalies = df[df["is_anomaly"] == 1]

plt.scatter(normal["amount"], normal["distance_from_home_km"], alpha=0.5, label="Normal")
plt.scatter(anomalies["amount"], anomalies["distance_from_home_km"], alpha=0.8, label="Anomaly")
plt.title("Anomalías por monto y distancia")
plt.xlabel("Monto")
plt.ylabel("Distancia desde ubicación habitual")
plt.legend()
plt.show()


In [ ]:
df.sort_values("anomaly_score").head(15)

## Conclusión

Isolation Forest es útil cuando no existen labels confiables de fraude. Este enfoque puede aplicarse a:

- Transacciones sospechosas
- Logins anómalos
- Consumo inusual de APIs
- Cambios raros de comportamiento financiero
